In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()


Q1 - How many lesson pages?

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)    

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [17]:
question = 'How does the agentic loop keep calling the model until it stops?'

Q2 - What's the filename of the first result?

In [21]:
search_results = index.search(
    question,
    # boost_dict={'question': 2.0, 'section': 0.5},
    # filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results[0]

{'start': 4000,
 'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model reaso

In [6]:
from rag_helper import RAGBase

In [7]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [8]:
index = build_index(documents)
assistant = RAGBase(index, openai_client)

In [9]:
search_results = assistant.rag(question)[2]
print(search_results)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Q3 - How many input (prompt) tokens did we send to the model for this request?

In [10]:
input_tokens = assistant.rag(question)[1]
print(input_tokens)

7111


In [11]:
answer = assistant.rag(question)[0]
print(answer)

The loop keeps calling the model by:

1. Sending the full message history to the model.
2. Checking the response for any `function_call` items.
3. Running those tool calls and appending the results to `messages`.
4. Repeating the API call in a `while True` loop.
5. Stopping only when the model returns no function calls.

In the code, the key flag is:

```python
has_function_calls = False
...
if item.type == "function_call":
    has_function_calls = True
...
if has_function_calls == False:
    break
```

So: **no function calls on a turn means the agent is done**.


In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

Q4 - How many chunks do you get?

In [13]:
len(chunks)

295

In [14]:
index = build_index(chunks)
assistant = RAGBase(index, openai_client)

In [15]:
answer = assistant.rag(question)[0]
print(answer)

The loop keeps calling the model inside a `while True` loop. Each turn:

1. It sends the current `messages` to the model.
2. It checks the model output for any `function_call`.
3. If there was a function call, it runs the tool, appends the tool result to `messages`, and continues.
4. If there were no function calls in that turn, it `break`s out of the loop.

So the stop condition is: **no function calls in the model’s response**.


Q5 - How many fewer input tokens does the chunked version send?

In [16]:
input_tokens = assistant.rag(question)[1]
print(input_tokens)

2294


In [17]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
def search(query):
    # boost_dict = {"question": 3.0, "section": 0.5}
    # filter_dict = {"course": "llm-zoomcamp"}
    index = build_index(chunks)
    return index.search(
        query,
        num_results=5
        # boost_dict=boost_dict,
        # filter_dict=filter_dict
    )

In [22]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course content for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course content."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [25]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [26]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG??",
    callback=callback,
)

-> Response received


-> Response received


In [27]:
result.cost

CostInfo(input_cost=Decimal('0.0062535'), output_cost=Decimal('0.003195'), total_cost=Decimal('0.0094485'))

In [28]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG??', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG agentic loop plain RAG course"}', call_id='call_KljmnnIkUt4xZqo2gm26FiVU', name='search', type='function_call', id='fc_0a5b292560555e08006a2848817fe081a3b316e3024da116fe', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agent

In [29]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


In [30]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop differ with different tools?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop different tools tool use agentic loop course"}', call_id='call_auuCmirvC8HFpZrxc8ANlB2u', name='search', type='function_call', id='fc_0502ad06e5d67847006a2848d8596881a29f0dd252b9807700', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{